# CNN

### Problem Statement
- Hand Written Digit Recognition

### Import Required Packages

In [1]:
import numpy as np
import torch
import torch.nn as nn

### Use The Required Device

In [2]:
device = ""
if torch.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"using device = {device}")

using device = cpu


### Get The Data

In [5]:
from torchvision import datasets
from torchvision import transforms

# create a transformer to convert the data into tensor
transformer = transforms.ToTensor()

In [8]:
# download the training data
# root : the loaction to download the data
# train : whether to download the train or test data
# download : whether to download and store the data on disk
# transform : used to convert the data into tensor
train_dataset = datasets.MNIST(
    root ="./data",train=True, download=True ,transform = transformer
)

100%|██████████| 9.91M/9.91M [00:03<00:00, 2.71MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 120kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 848kB/s] 
100%|██████████| 4.54k/4.54k [00:00<?, ?B/s]


In [9]:
# download the test dataset
test_dataset = datasets.MNIST(
    root="./data",train=False,download=True,transform=transformer
)

In [10]:
train_dataset

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [11]:
test_dataset

Dataset MNIST
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: ToTensor()

### Create Data Loader

In [12]:
from torch.utils.data import DataLoader

# create a train loader
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)

# create a test loader
test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False)

### Model Definition

In [13]:
# create the model by adding required layers
# input for the CNN will be 64 images of 28 x 28 pixels in a single batch
# at a time the CNN will receive an image of 28 x 28 pixels
# the input shape here will be 1 x 28 x 28

# the max pool will reduce the feature mao from 32 to 7
# 32 // 2 = 16 - 2 = 14 / 2 = 7

In [14]:
model = nn.Sequential(
    # add the convolution2d layer
    # in_channels = 1: at a time only one image will be fed to the CNN for processing
    # out_channels = 16 : number of feature to be extracted ( the CNN will create 16 number of kernels , each one of 3 x 3 size)
    # kernel_size = 3 : we will be using kernel 3 x 3 size
    # padding = 1 : add the padding on all the borders
    # stride 1 : move to the right side by one column and down by one row

    nn.Conv2d(in_channels=1, out_channels=16,kernel_size=3,padding=1,stride=1),

    # configure the ReLu activation function
    nn.ReLU(),

    # add max pooling layer
    nn.MaxPool2d(kernel_size=2,stride=2),

    # add one more convolution layer
    nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1, stride=1),

    # configure ReLu activation function
    nn.ReLU(),

    # add max pooling layer
    nn.MaxPool2d(kernel_size=2, stride=2),

    # add the flatten layer
    nn.Flatten(),

    # add the hidden layer with 128 neurons
    nn.Linear(in_features=32*7*7,out_features=128),

    # add ReLu on the hidden layer
    nn.ReLU(),

    # add the output layer
    # since there are 10 digits the output layer will have 10 neurons
    nn.Linear(in_features=128, out_features=10)

)
# move the model to device
model = model.to(device)

In [15]:
model

Sequential(
  (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=1568, out_features=128, bias=True)
  (8): ReLU()
  (9): Linear(in_features=128, out_features=10, bias=True)
)

### Define The Hyperparameters

In [16]:
# number of epochs
epochs = 5
# loss function
# since there are 10 classes , we can Not use BCEntropyLoss()

loss_function = nn.CrossEntropyLoss()

# learning rate
learning_rate = 0.001

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

### Training Loop

In [17]:
# collect all the losses
losses = []

for epoch in range(epochs):
    # enable the training mode
    model.train()

    # get the total loss for all the batches
    running_loss = 0

    # pass the 64 images and labels in batches  to the model
    for images,labels in train_loader:
        # move the images and labels to the device
        images = images.to(device)
        labels = labels.to(device)

        # clear the previous gradients
        optimizer.zero_grad()

        # predict the values for the training set
        predictions = model(images)

        # calculate the loss
        loss = loss_function(predictions, labels)

        # calculate the loss gradients
        loss.backward()

        # optimize the parameters
        optimizer.step()

        # add the loss to the running_loss
        running_loss += loss.item()

    # print the progress
    print(f"epoch = {epoch}, error = {running_loss/(len(train_loader)):.2f}")


epoch = 0, error = 0.23
epoch = 1, error = 0.06
epoch = 2, error = 0.04
epoch = 3, error = 0.03
epoch = 4, error = 0.02


### evaluate the model

In [19]:
# enable the evaluation mode
model.eval()

Sequential(
  (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=1568, out_features=128, bias=True)
  (8): ReLU()
  (9): Linear(in_features=128, out_features=10, bias=True)
)

In [20]:
# define the stats
correct = 0
total = 0

In [21]:
# Disable  updating the weights
with torch.no_grad():

    # pass the test images to the model batch by batch
    for images , labels in test_loader:

        # move the images and labels to the device
        images = images.to(device)
        labels = labels.to(device)

        # pass the images to the model
        # for example :
        # -1 prediction = [0.001,0.600,0.004....]
        predictions = model(images)
        # print(predictions)

        # find the final class by taking the max value from the predictions
        predictions = torch.argmax(predictions, dim=1)
        # print(predictions)

        # update the total number of classification
        total += labels.size(0)

        # find the number of correct predictions
        correct += (predictions == labels).sum().item()

# calculate the accuracy
accuracy = (correct / total) * 100

print(f"accuracy = {accuracy:.2f}%")

accuracy = 99.07%


### Save The Model

In [22]:
torch.save(model.state_dict(),"mnist_model.pth")

### How the softmax works

In [23]:
# create an array of numbers
numbers = np.array([1, 2, 3, 10, 4, 8, 19, 4])

# find the the index of largest value
np.argmax(numbers)

np.int64(6)